# 🛍️ Customer Segmentation — Interactive GUI
**Egyptian Chinese University (ECU) — **  
Dataset: Mall Customer Segmentation | Algorithm: K-Means & Agglomerative Clustering

---

In [ ]:
# ============================================================
# CELL 1: Install & Import
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

print('✅ All libraries imported successfully!')

In [ ]:
# ============================================================
# CELL 2: Load Dataset
# ============================================================
df = pd.read_csv('Mall_Customers.csv')
X  = df[['Annual Income (k$)', 'Spending Score (1-100)']].values

# Pre-compute clustering results
kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(X)

agg = AgglomerativeClustering(n_clusters=5, linkage='ward')
df['Agg_Cluster'] = agg.fit_predict(X)

kmeans_sil = silhouette_score(X, df['KMeans_Cluster'])
agg_sil    = silhouette_score(X, df['Agg_Cluster'])

COLORS       = ['#E24B4A','#1D9E75','#378ADD','#BA7517','#9B59B6']
CLUSTER_TAGS = ['Careful customers','Impulsive buyers','Average customers','Potential savers','VIP targets']
CLUSTER_DESC = [
    'Low Income  · Low Spending',
    'Low Income  · High Spending',
    'Medium Income · Medium Spending',
    'High Income · Low Spending',
    'High Income · High Spending',
]

print(f'✅ Dataset loaded — {len(df)} customers, 5 clusters')
print(f'   K-Means Silhouette  : {kmeans_sil:.4f}')
print(f'   Agglomerative Sil.  : {agg_sil:.4f}')
df.head()

---
## 📊 Tab 1 — Optimal K Finder (Elbow + Silhouette)

In [ ]:
# ============================================================
# CELL 3: Optimal K Finder
# ============================================================
inertia, sil_scores = [], []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km.fit(X)
    inertia.append(km.inertia_)
    if k >= 2:
        sil_scores.append(silhouette_score(X, km.labels_))

k_slider = widgets.IntSlider(
    value=5, min=2, max=10, step=1,
    description='Highlight K:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='420px')
)
out_k = widgets.Output()

def plot_optimal_k(k):
    with out_k:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle('Finding Optimal K', fontsize=15, fontweight='bold', y=1.01)

        # Elbow
        axes[0].plot(list(K_range), inertia, 'o-', color='#378ADD', lw=2, ms=8)
        axes[0].axvline(x=k, color='#E24B4A', ls='--', lw=1.8, label=f'K = {k}')
        axes[0].fill_between(list(K_range), inertia, alpha=0.08, color='#378ADD')
        axes[0].set_title('Elbow Method — Inertia (WCSS)', fontsize=13, fontweight='bold')
        axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
        axes[0].set_ylabel('Inertia', fontsize=12)
        axes[0].legend(fontsize=11)
        axes[0].set_xticks(list(K_range))

        # Silhouette
        sil_x = list(range(2, 11))
        bar_colors = ['#E24B4A' if x == k else '#1D9E75' for x in sil_x]
        axes[1].bar(sil_x, sil_scores, color=bar_colors, edgecolor='white', linewidth=0.5)
        axes[1].set_title('Silhouette Scores', fontsize=13, fontweight='bold')
        axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
        axes[1].set_ylabel('Silhouette Score', fontsize=12)
        axes[1].set_xticks(sil_x)
        for i, (x, s) in enumerate(zip(sil_x, sil_scores)):
            axes[1].text(x, s + 0.005, f'{s:.3f}', ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.show()
        best_sil_k = sil_x[np.argmax(sil_scores)]
        print(f'   Best Silhouette Score: K={best_sil_k} ({max(sil_scores):.4f})')
        print(f'   Currently highlighted: K={k} | Inertia={inertia[k-1]:,.0f} | Sil={sil_scores[k-2]:.4f}')

widgets.interactive_output(plot_optimal_k, {'k': k_slider})
display(widgets.VBox([
    widgets.HTML('<b style="font-size:14px">🔍 Drag to explore different K values</b>'),
    k_slider,
    out_k
]))
plot_optimal_k(5)

---
## 🔵 Tab 2 — K-Means Scatter Explorer

In [ ]:
# ============================================================
# CELL 4: K-Means Interactive Scatter
# ============================================================
xaxis_dd = widgets.Dropdown(
    options=[('Annual Income (k$)', 'Annual Income (k$)'), ('Age', 'Age')],
    value='Annual Income (k$)', description='X-axis:',
    style={'description_width': '60px'}, layout=widgets.Layout(width='260px')
)
yaxis_dd = widgets.Dropdown(
    options=[('Spending Score (1-100)', 'Spending Score (1-100)'), ('Annual Income (k$)', 'Annual Income (k$)')],
    value='Spending Score (1-100)', description='Y-axis:',
    style={'description_width': '60px'}, layout=widgets.Layout(width='260px')
)
show_centroids = widgets.Checkbox(value=True, description='Show centroids', layout=widgets.Layout(width='160px'))
out_scatter = widgets.Output()

def plot_kmeans_scatter(x_col, y_col, centroids):
    with out_scatter:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 6))
        for c in range(5):
            mask = df['KMeans_Cluster'] == c
            ax.scatter(df[mask][x_col], df[mask][y_col],
                       c=COLORS[c], label=f'Cluster {c} — {CLUSTER_TAGS[c]}',
                       s=80, edgecolors='white', linewidth=0.5, alpha=0.88)
        if centroids and x_col == 'Annual Income (k$)' and y_col == 'Spending Score (1-100)':
            cx = kmeans.cluster_centers_[:, 0]
            cy = kmeans.cluster_centers_[:, 1]
            ax.scatter(cx, cy, c='black', marker='X', s=220, zorder=5, label='Centroids')
        ax.set_xlabel(x_col, fontsize=12)
        ax.set_ylabel(y_col, fontsize=12)
        ax.set_title(f'K-Means: {y_col} vs {x_col}', fontsize=14, fontweight='bold')
        ax.legend(fontsize=10, bbox_to_anchor=(1.01, 1), loc='upper left')
        ax.grid(alpha=0.2)
        plt.tight_layout()
        plt.show()

widgets.interactive_output(plot_kmeans_scatter,
    {'x_col': xaxis_dd, 'y_col': yaxis_dd, 'centroids': show_centroids})
display(widgets.VBox([
    widgets.HTML('<b style="font-size:14px">🎛️ Customize the scatter plot</b>'),
    widgets.HBox([xaxis_dd, yaxis_dd, show_centroids]),
    out_scatter
]))
plot_kmeans_scatter('Annual Income (k$)', 'Spending Score (1-100)', True)

---
## 🌳 Tab 3 — Agglomerative + Dendrogram

In [ ]:
# ============================================================
# CELL 5: Agglomerative + Dendrogram
# ============================================================
cut_slider = widgets.FloatSlider(
    value=200, min=50, max=400, step=10,
    description='Cut line:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)
out_dend = widgets.Output()

linked = linkage(X, method='ward')

def plot_dendrogram(cut):
    with out_dend:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Dendrogram
        dendrogram(linked, ax=axes[0], truncate_mode='lastp', p=30,
                   leaf_font_size=9, show_contracted=True, color_threshold=cut)
        axes[0].axhline(y=cut, color='#E24B4A', ls='--', lw=2,
                        label=f'Cut = {cut:.0f}')
        n_clusters = int(np.sum(linked[:, 2] > cut)) + 1 if cut < linked[-1, 2] else 1
        axes[0].set_title(f'Dendrogram — Ward Linkage (→ {n_clusters} clusters)', fontsize=13, fontweight='bold')
        axes[0].set_xlabel('Customer samples', fontsize=11)
        axes[0].set_ylabel('Euclidean distance', fontsize=11)
        axes[0].legend(fontsize=11)

        # Agglomerative scatter
        for c in range(5):
            mask = df['Agg_Cluster'] == c
            axes[1].scatter(df[mask]['Annual Income (k$)'], df[mask]['Spending Score (1-100)'],
                            c=COLORS[c], label=f'Cluster {c} — {CLUSTER_TAGS[c]}',
                            s=80, edgecolors='white', linewidth=0.5, alpha=0.88)
        axes[1].set_title(f'Agglomerative (Sil: {agg_sil:.4f})', fontsize=13, fontweight='bold')
        axes[1].set_xlabel('Annual Income (k$)', fontsize=11)
        axes[1].set_ylabel('Spending Score (1-100)', fontsize=11)
        axes[1].legend(fontsize=9, bbox_to_anchor=(1.01, 1), loc='upper left')
        axes[1].grid(alpha=0.2)

        plt.tight_layout()
        plt.show()

widgets.interactive_output(plot_dendrogram, {'cut': cut_slider})
display(widgets.VBox([
    widgets.HTML('<b style="font-size:14px">✂️ Drag cut line to change number of clusters</b>'),
    cut_slider,
    out_dend
]))
plot_dendrogram(200)

---
## 🏷️ Tab 4 — Segment Profiles Dashboard

In [ ]:
# ============================================================
# CELL 6: Segment Profiles Dashboard
# ============================================================
algo_toggle = widgets.ToggleButtons(
    options=['K-Means', 'Agglomerative'],
    value='K-Means',
    description='Algorithm:',
    style={'description_width': '80px', 'button_width': '130px'}
)
out_profiles = widgets.Output()

def plot_profiles(algo):
    col = 'KMeans_Cluster' if algo == 'K-Means' else 'Agg_Cluster'
    with out_profiles:
        clear_output(wait=True)
        summary = df.groupby(col)[['Age','Annual Income (k$)','Spending Score (1-100)']].mean().round(1)
        counts  = df[col].value_counts().sort_index()

        fig, axes = plt.subplots(2, 3, figsize=(15, 9))
        fig.suptitle(f'{algo} — Segment Profiles', fontsize=15, fontweight='bold')

        # Bar: cluster sizes
        axes[0,0].bar(counts.index, counts.values, color=COLORS, edgecolor='white', linewidth=0.5)
        axes[0,0].set_title('Cluster sizes', fontweight='bold')
        axes[0,0].set_xlabel('Cluster'); axes[0,0].set_ylabel('Count')
        for i, v in enumerate(counts.values):
            axes[0,0].text(i, v+0.5, str(v), ha='center', fontsize=11, fontweight='bold')

        # Bar: avg income
        axes[0,1].bar(summary.index, summary['Annual Income (k$)'], color=COLORS, edgecolor='white')
        axes[0,1].set_title('Avg annual income (k$)', fontweight='bold')
        axes[0,1].set_xlabel('Cluster'); axes[0,1].set_ylabel('Income (k$)')

        # Bar: avg spending
        axes[0,2].bar(summary.index, summary['Spending Score (1-100)'], color=COLORS, edgecolor='white')
        axes[0,2].set_title('Avg spending score', fontweight='bold')
        axes[0,2].set_xlabel('Cluster'); axes[0,2].set_ylabel('Score')

        # Scatter: income vs spending (colored)
        for c in range(5):
            mask = df[col] == c
            axes[1,0].scatter(df[mask]['Annual Income (k$)'], df[mask]['Spending Score (1-100)'],
                              c=COLORS[c], s=60, alpha=0.8, edgecolors='white', lw=0.4)
        axes[1,0].set_title('Income vs spending', fontweight='bold')
        axes[1,0].set_xlabel('Income (k$)'); axes[1,0].set_ylabel('Spending score')

        # Pie: cluster share
        axes[1,1].pie(counts.values, labels=[f'C{i}' for i in range(5)],
                      colors=COLORS, autopct='%1.1f%%', startangle=90,
                      wedgeprops={'edgecolor':'white','linewidth':1.5})
        axes[1,1].set_title('Cluster share (%)', fontweight='bold')

        # Box: spending per cluster
        data_box = [df[df[col]==c]['Spending Score (1-100)'].values for c in range(5)]
        bp = axes[1,2].boxplot(data_box, patch_artist=True, notch=True,
                               medianprops={'color':'white','linewidth':2})
        for patch, color in zip(bp['boxes'], COLORS):
            patch.set_facecolor(color); patch.set_alpha(0.85)
        axes[1,2].set_title('Spending score distribution', fontweight='bold')
        axes[1,2].set_xlabel('Cluster'); axes[1,2].set_ylabel('Spending score')
        axes[1,2].set_xticklabels([f'C{i}' for i in range(5)])

        plt.tight_layout()
        plt.show()

        print('\n📋 Cluster Summary Table:')
        summary['Count'] = counts.values
        summary['Segment'] = [CLUSTER_TAGS[i] for i in range(5)]
        print(summary[['Count','Age','Annual Income (k$)','Spending Score (1-100)','Segment']].to_string())

widgets.interactive_output(plot_profiles, {'algo': algo_toggle})
display(widgets.VBox([algo_toggle, out_profiles]))
plot_profiles('K-Means')

---
## ⚖️ Tab 5 — Algorithm Comparison

In [ ]:
# ============================================================
# CELL 7: Side-by-Side Comparison
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for c in range(5):
    m = df['KMeans_Cluster'] == c
    axes[0].scatter(df[m]['Annual Income (k$)'], df[m]['Spending Score (1-100)'],
                    c=COLORS[c], label=f'C{c} — {CLUSTER_TAGS[c]}', s=70,
                    edgecolors='white', linewidth=0.4, alpha=0.88)
cx = kmeans.cluster_centers_[:, 0]; cy = kmeans.cluster_centers_[:, 1]
axes[0].scatter(cx, cy, c='black', marker='X', s=220, zorder=5, label='Centroids')
axes[0].set_title(f'K-Means  (Silhouette: {kmeans_sil:.4f}) ✅ Winner', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Annual Income (k$)'); axes[0].set_ylabel('Spending Score (1-100)')
axes[0].legend(fontsize=9, loc='upper left'); axes[0].grid(alpha=0.2)

for c in range(5):
    m = df['Agg_Cluster'] == c
    axes[1].scatter(df[m]['Annual Income (k$)'], df[m]['Spending Score (1-100)'],
                    c=COLORS[c], label=f'C{c} — {CLUSTER_TAGS[c]}', s=70,
                    edgecolors='white', linewidth=0.4, alpha=0.88)
axes[1].set_title(f'Agglomerative  (Silhouette: {agg_sil:.4f})', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Annual Income (k$)'); axes[1].set_ylabel('Spending Score (1-100)')
axes[1].legend(fontsize=9, loc='upper left'); axes[1].grid(alpha=0.2)

plt.suptitle('K-Means vs Agglomerative Clustering', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('=' * 52)
print('        ALGORITHM COMPARISON SUMMARY')
print('=' * 52)
print(f'  K-Means Silhouette       : {kmeans_sil:.4f}  ✅')
print(f'  Agglomerative Silhouette : {agg_sil:.4f}')
print('=' * 52)
print('  Winner: K-Means gives better-defined clusters')

---
## 🔮 Tab 6 — Predict Customer Segment

In [ ]:
# ============================================================
# CELL 8: Predict New Customer Segment
# ============================================================
income_slider = widgets.IntSlider(
    value=70, min=1, max=150, step=1,
    description='Income (k$):',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='420px')
)
spending_slider = widgets.IntSlider(
    value=75, min=1, max=100, step=1,
    description='Spending score:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='420px')
)
algo_pred = widgets.ToggleButtons(
    options=['K-Means', 'Agglomerative'],
    description='Algorithm:',
    style={'description_width': '80px', 'button_width': '120px'}
)
out_pred = widgets.Output()

CENTROIDS_KM  = kmeans.cluster_centers_
CENTROIDS_AGG = df.groupby('Agg_Cluster')[['Annual Income (k$)','Spending Score (1-100)']].mean().values

STRATEGIES = [
    '💰 Price-sensitive promotions and budget-friendly deals.',
    '🛍️ Flash sales, limited-time offers and loyalty programs.',
    '😐 Standard marketing campaigns and seasonal promotions.',
    '💎 Exclusive high-value offers to unlock spending potential.',
    '⭐ Premium VIP experiences, early access and personalized services.',
]

def predict_segment(income, spending, algo):
    with out_pred:
        clear_output(wait=True)
        point = np.array([[income, spending]])
        if algo == 'K-Means':
            cluster = kmeans.predict(point)[0]
            centers = CENTROIDS_KM
        else:
            dists = np.linalg.norm(CENTROIDS_AGG - point, axis=1)
            cluster = int(np.argmin(dists))
            centers = CENTROIDS_AGG

        fig, ax = plt.subplots(figsize=(9, 6))
        for c in range(5):
            col_key = 'KMeans_Cluster' if algo == 'K-Means' else 'Agg_Cluster'
            mask = df[col_key] == c
            ax.scatter(df[mask]['Annual Income (k$)'], df[mask]['Spending Score (1-100)'],
                       c=COLORS[c], s=55, alpha=0.5, edgecolors='white', linewidth=0.3)

        # Highlight predicted cluster
        col_key = 'KMeans_Cluster' if algo == 'K-Means' else 'Agg_Cluster'
        mask = df[col_key] == cluster
        ax.scatter(df[mask]['Annual Income (k$)'], df[mask]['Spending Score (1-100)'],
                   c=COLORS[cluster], s=80, alpha=1.0, edgecolors='white', linewidth=0.5,
                   label=f'Cluster {cluster} — {CLUSTER_TAGS[cluster]}')

        # New customer point
        ax.scatter(income, spending, c='black', s=280, marker='*', zorder=10, label='New customer')
        ax.annotate(f'  You are here\n  Cluster {cluster}', (income, spending),
                    fontsize=11, fontweight='bold', color=COLORS[cluster],
                    xytext=(income+5, spending+3))

        ax.set_xlabel('Annual Income (k$)', fontsize=12)
        ax.set_ylabel('Spending Score (1-100)', fontsize=12)
        ax.set_title(f'{algo} — Customer Prediction', fontsize=14, fontweight='bold')
        ax.legend(fontsize=10); ax.grid(alpha=0.2)
        plt.tight_layout()
        plt.show()

        print('─' * 50)
        print(f'  📍 Input     : Income=${income}k  |  Score={spending}')
        print(f'  🏷️  Segment   : Cluster {cluster} — {CLUSTER_TAGS[cluster]}')
        print(f'  📊 Profile   : {CLUSTER_DESC[cluster]}')
        print(f'  📢 Strategy  : {STRATEGIES[cluster]}')
        print('─' * 50)

widgets.interactive_output(predict_segment,
    {'income': income_slider, 'spending': spending_slider, 'algo': algo_pred})

display(widgets.VBox([
    widgets.HTML('<b style="font-size:14px">🔮 Enter customer profile to predict their segment</b>'),
    income_slider,
    spending_slider,
    algo_pred,
    out_pred
]))
predict_segment(70, 75, 'K-Means')

---
## 📋 Tab 7 — Raw Data Explorer

In [ ]:
# ============================================================
# CELL 9: Data Explorer
# ============================================================
cluster_filter = widgets.SelectMultiple(
    options=[('All clusters', -1)] + [(f'Cluster {i} — {CLUSTER_TAGS[i]}', i) for i in range(5)],
    value=[-1],
    description='Filter:',
    rows=6,
    style={'description_width': '55px'},
    layout=widgets.Layout(width='320px')
)
n_rows_slider = widgets.IntSlider(
    value=10, min=5, max=50, step=5,
    description='Show rows:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='360px')
)
out_data = widgets.Output()

def show_data(clusters, n_rows):
    with out_data:
        clear_output(wait=True)
        if -1 in clusters:
            filtered = df.copy()
        else:
            filtered = df[df['KMeans_Cluster'].isin(clusters)].copy()

        filtered['Segment'] = filtered['KMeans_Cluster'].map(
            {i: CLUSTER_TAGS[i] for i in range(5)})

        display(filtered[['CustomerID','Gender','Age','Annual Income (k$)',
                           'Spending Score (1-100)','KMeans_Cluster','Segment']].head(n_rows))
        print(f'\n   Showing {min(n_rows, len(filtered))} of {len(filtered)} customers')

widgets.interactive_output(show_data, {'clusters': cluster_filter, 'n_rows': n_rows_slider})
display(widgets.VBox([
    widgets.HTML('<b style="font-size:14px">📋 Browse the clustered dataset</b>'),
    widgets.HBox([cluster_filter, n_rows_slider]),
    out_data
]))
show_data((-1,), 10)